In [3]:
import random
import math

# Before Training:

In [4]:
file = open("aspell.txt")
lines = file.readlines()
file.close()

word = []

for l in lines: 
    correct, typos = l.split(":")
    typo_list = typos.split()
    for typo in typo_list:
        word.append((correct, typo))

* Normalizing :lowercase everything and drop different size lengths

In [5]:
word = [(correct.lower(), typo.lower())for(correct,typo) in word]
lengths = [(c,t) for (c,t) in word if len(c) == len(t)]


# Split

In [6]:
random.seed(42)
random.shuffle(lengths)

test_num = int(len(lengths) * 0.2)
test_pairs = lengths[:test_num]
train_pairs = lengths[test_num:]


print("Train: ",len(train_pairs), "Test: ", len(test_pairs))
print(train_pairs)

Train:  173 Test:  43
[('anatomy', 'anonomy'), ('raccoon', 'reccona'), ('amorphous', 'amourfous'), ('announce', 'annuncio'), ('commercials', 'commerciasl'), ('media', 'midia'), ('metaphysical', 'pataphysical'), ('engine', 'engins'), ('incomplete', 'incompleet'), ('running', 'runnung'), ('dynamic', 'dymatic'), ('version', 'verison'), ('ecstasy', 'ecstacy'), ('flux', 'fluk'), ('battalion', 'batallion'), ('desiccate', 'dessicate'), ('supersede', 'supercede'), ('monkey', 'minkay'), ('despair', 'dispair'), ('nevada', 'nevade'), ('signal', 'singal'), ('immanent', 'immenant'), ('unnatural', 'unaturral'), ('combo', 'cumba'), ('grammar', 'grammer'), ('maintenance', 'maintanence'), ('intelligent', 'intelegnent'), ('government', 'gobernment'), ('recommend', 'reccomend'), ('scrabble', 'scrabdle'), ('weaken', 'wicken'), ('cheat', 'cheet'), ('problematic', 'proplematic'), ('pronunciation', 'pronensiation'), ('elegant', 'elligit'), ('critique', 'ctitique'), ('organize', 'organise'), ('insistent', 'in

# Emisison Probabilities

In [7]:
em_counts = {}

for correct, typed in train_pairs:
    for i in range(len(correct)):
        intended = correct[i]
        typed_char = typed[i]
        if intended not in em_counts:
            em_counts[intended] = {}
        if typed_char not in em_counts[intended]:
            em_counts[intended][typed_char] = 0
        em_counts[intended][typed_char] += 1

* Smoothing

In [8]:
alph = list("abcdefghijklmnopqrstuvwxyz")
v = len(alph)

em_probs = {}

for intended in em_counts:
    em_probs[intended] = {}
    total = sum(em_counts[intended].values())
    for char in alph:
        count = em_counts[intended].get(char, 0)
        em_probs[intended][char] = (count + 1) / (total + v)

# printing to test 
print(em_probs["a"]["a"])
print(em_probs["a"]["z"])
       

0.5384615384615384
0.007692307692307693


testing the Laplace smoothing with likely intented characters and unlikely intended characters. We see a .00769 chance a letter z is in the place of the correct letter a, this doesnt seem to show up in the training set at all( with the understanding that I havnt manually scanned all the words but after a brief skim i didnt seem to see any) but we want a non zero probability so this looks good. 

# Transitions Probabilities 

In [9]:
trans_counts = {}
start_counts = {}
end_counts = {}

for correct, typed in train_pairs:
    start_let = correct[0]
    start_counts[start_let] = start_counts.get(start_let, 0) + 1

    end_let = correct[-1]
    end_counts[end_let] = end_counts.get(end_let, 0) + 1

    for i in range(len(correct) - 1):
        curr = correct[i]
        next = correct[i + 1]
        if curr not in trans_counts:
            trans_counts[curr] = {}
        trans_counts[curr][next] = trans_counts[curr].get(next, 0) + 1
print("# of times each letter starts a word:")
print(start_counts)
print()
print("# of times a letter occurs after a:")
print(trans_counts["a"])

# of times each letter starts a word:
{'a': 8, 'r': 5, 'c': 16, 'm': 11, 'e': 13, 'i': 14, 'd': 23, 'v': 1, 'f': 5, 'b': 12, 's': 18, 'n': 3, 'u': 2, 'g': 4, 'w': 4, 'p': 12, 'o': 5, 't': 6, 'h': 4, 'y': 2, 'k': 3, 'l': 1, 'q': 1}

# of times a letter occurs after a:
{'n': 18, 't': 19, 'c': 4, 'm': 8, 'l': 15, 'p': 2, 's': 5, 'i': 3, 'd': 3, 'r': 8, 'b': 3, 'k': 2, 'u': 3, 'z': 1, 'v': 1, 'y': 1, 'j': 2, 'f': 2, 'g': 2}


- I chose a for convienence just to make sure everything looks relativley correct. I ran it with a few other letters too.
- I also see that there is actually 1 occurance of 'z' appearing right after 'a'

* converting transition counts to smoothing probs

In [10]:
all_words = len(train_pairs)
start_probs = {}
end_probs = {}
trans_probs = {}

for a in alph:
    count = start_counts.get(a,0)
    start_probs[a] = (count + 1) / (all_words + v)

for a in alph:
    count = end_counts.get(a,0)
    end_probs[a] = (count + 1) / (all_words + v)

for curr in alph:
    trans_probs[curr] = {}
    total = sum(trans_counts.get(curr, {}).values())
    for next in alph:
        count = trans_counts.get(curr, {}).get(next, 0)
        trans_probs[curr][next] = (count + 1) / (total + v)

print(start_probs["e"])
print(end_probs["e"])
print(start_probs["a"])
print(end_probs["a"])
print(trans_probs["a"]["n"])
print(trans_probs["a"]["z"])

0.07035175879396985
0.21608040201005024
0.04522613065326633
0.01507537688442211
0.1484375
0.015625


Note: Added start and end probs for 'e' after running 'correcting user txt' code cells to verify my assumption that 'e' has a higher start and end probabilie than 'a' (see relavance at that cell)

Using random letters to test: After smoothing ->
for transition probability for 'n' after 'a' we see .1532, this seems correct given what we saw earlier where 'n' was the most common letter after 'a' occuring 18 times. This is the same for 'z' after 'a', looking at the occurances above 'z' after 'a' is one of the lowest occurances sharing the space with 'g' and 'k'

# Viterbi Decoding

In [ ]:
def viterbi(typed_word):
    n = len(typed_word)
    V = [{} for _ in range(n)]      
    back = [{} for _ in range(n)]  

    # Base case: 
    typed_letter = typed_word[0]
    for letter in alph:
        log_start = math.log(start_probs[letter])
        log_emit = math.log(em_probs[letter][typed_letter])
        V[0][letter] = log_start + log_emit

    # Recurrence
    for i in range(1, n):
        typed_letter = typed_word[i]
        for letter in alph:
            log_emit = math.log(em_probs[letter][typed_letter])

            best_score = None
            best_prev = None
            for prev_letter in alph:
                log_trans = math.log(trans_probs[prev_letter][letter])
                score = V[i - 1][prev_letter] + log_trans + log_emit
                if best_score is None or score > best_score:
                    best_score = score
                    best_prev = prev_letter

            V[i][letter] = best_score
            back[i][letter] = best_prev

    # END probability
    best_final_score = None
    best_final_letter = None
    for letter in alph:
        score = V[n - 1][letter] + math.log(end_probs[letter])
        if best_final_score is None or score > best_final_score:
            best_final_score = score
            best_final_letter = letter

    # Reconstructing from backwards
    path = [best_final_letter]
    for i in range(n - 1, 0, -1):
        prev_letter = back[i][path[-1]]
        path.append(prev_letter)
    path.reverse()

    return "".join(path), best_final_score


word, score = viterbi("hte")
print(word, score)

hte -12.310916700260778


based on my functon viterbi() decided that "hte" is actually the correct word. becasue of the smal training set the function actually doesnt have enough example where 't' was accidentally typed as 'h'. however 'h' being typed for words that start with 'h' (ie no typo) has a .34, so thats what its relying on

# Correcting user txt

In [12]:
def fix_text(text):
    words = text.split()
    corrected_words = []

    for word in words:
        clean_word = word.lower()
        clean_word = "".join(ch for ch in clean_word if ch in alph)

        if len(clean_word) == 0:
            corrected_words.append(word)   
            continue

        corrected, score = viterbi(clean_word)
        corrected_words.append(corrected)

    return " ".join(corrected_words)


print(fix_text(""))
print(fix_text("Hte dog, ran"))
print(fix_text("a"))


hte don ran
e


"hte" resulted as expected but 'e' was not expected. I would assume the data would have an instance of 'a' more likely being a full word than 'e' but now actually looking at the training set I dont think I see any instances of just 'a', but 'e' has both a higher starting and ending probabilitie than 'a' so this makes sence ( additonal checks for starting and ending probabilities for 'e' were added after I ran this cell to verify my assumption)

# Confidence threshold

In [13]:
def same_score(typed_word):
    word = len(typed_word)
    score = math.log(start_probs[typed_word[0]])
    score += math.log(em_probs[typed_word[0]][typed_word[0]])
    for i in range(1, word):
        score += math.log(trans_probs[typed_word[i-1]][typed_word[i]])
        score += math.log(em_probs[typed_word[i]][typed_word[i]])
    score += math.log(end_probs[typed_word[-1]])
    return score

def threshold(typed_word, threshold):
    corrected, score = viterbi(typed_word)
    same = same_score(typed_word)

    if score - same > threshold:
        return corrected
    else:
        return typed_word


# testing thresholds

In [14]:
thresholds_to_try = [0, 1, 2, 3, 5]

for correct_word, typed_word in test_pairs[:5]:
    print(f"correct: {correct_word!r}")
    for t in thresholds_to_try:
        print("threshold =",t, ": ", threshold(correct_word, t))
    print()

    print(f"typed:   {typed_word!r}")
    for t in thresholds_to_try:
        print("threshold =",t, ": ", threshold(typed_word, t))
    print()

correct: 'colon'
threshold = 0 :  colon
threshold = 1 :  colon
threshold = 2 :  colon
threshold = 3 :  colon
threshold = 5 :  colon

typed:   'coaln'
threshold = 0 :  coaln
threshold = 1 :  coaln
threshold = 2 :  coaln
threshold = 3 :  coaln
threshold = 5 :  coaln

correct: 'naive'
threshold = 0 :  naive
threshold = 1 :  naive
threshold = 2 :  naive
threshold = 3 :  naive
threshold = 5 :  naive

typed:   'nieve'
threshold = 0 :  nieve
threshold = 1 :  nieve
threshold = 2 :  nieve
threshold = 3 :  nieve
threshold = 5 :  nieve

correct: 'caveats'
threshold = 0 :  caveats
threshold = 1 :  caveats
threshold = 2 :  caveats
threshold = 3 :  caveats
threshold = 5 :  caveats

typed:   'cravets'
threshold = 0 :  cravets
threshold = 1 :  cravets
threshold = 2 :  cravets
threshold = 3 :  cravets
threshold = 5 :  cravets

correct: 'efficacy'
threshold = 0 :  efficacy
threshold = 1 :  efficacy
threshold = 2 :  efficacy
threshold = 3 :  efficacy
threshold = 5 :  efficacy

typed:   'efficity'
thresho

In [15]:
def evaluate(threshold_value):
    total_words = 0
    correct_words = 0
    total_chars = 0
    correct_chars = 0
    misspelled_total = 0
    misspelled_fixed = 0
    correct_input_total = 0
    correct_input_broken = 0

    for correct_word, typed_word in test_pairs:
        output = threshold(typed_word, threshold_value)

        total_words += 1
        if output == correct_word:
            correct_words += 1

        total_chars += len(correct_word)
        for i in range(len(correct_word)):
            if i < len(output) and output[i] == correct_word[i]:
                correct_chars += 1

        if typed_word != correct_word:
            misspelled_total += 1
            if output == correct_word:
                misspelled_fixed += 1
        else:
            correct_input_total += 1
            if output != correct_word:
                correct_input_broken += 1

    return {
        "exact_word_acc": correct_words / total_words,
        "char_acc": correct_chars / total_chars,
        "success_rate": misspelled_fixed / misspelled_total if misspelled_total else 0,
        "false_rate": correct_input_broken / correct_input_total if correct_input_total else 0,
    }


for t in [0, 1, 2, 3, 5]:
    r = evaluate(t)
    exact = round(r["exact_word_acc"] * 100, 1)
    char = round(r["char_acc"] * 100, 1)
    success = round(r["success_rate"] * 100, 1)
    false = round(r["false_rate"] * 100, 1)
    print("t =", t, " exact =", r["exact_word_acc"] * 100, "% char =", r["char_acc"] * 100, "% success =", r["success_rate"] * 100, "% false =", r["false_rate"] * 100, "%")

t = 0  exact = 2.3255813953488373 % char = 76.67638483965014 % success = 2.380952380952381 % false = 100.0 %
t = 1  exact = 0.0 % char = 77.84256559766763 % success = 0.0 % false = 100.0 %
t = 2  exact = 0.0 % char = 78.134110787172 % success = 0.0 % false = 100.0 %
t = 3  exact = 2.3255813953488373 % char = 78.4256559766764 % success = 0.0 % false = 0.0 %
t = 5  exact = 2.3255813953488373 % char = 78.4256559766764 % success = 0.0 % false = 0.0 %


The metrics at first glance look pretty terrible, but its important to aknowlage that the training set is extremly low and sparse considering the task at hand. Earlier I had hand tested some words with different thresholds before looking over the spec and realizing that the assignment didnt want me to do it by hand and I found that it really didnt make many imporvments and at times actually incorrectly corrected already valid words, one that I did find intresting was the word 'liquefy' I tried a few versions 'liquify', 'liquufy', 'liqupfy' and it corrected both 'liquify'and 'liquufy' to 'liquefy' which looked off to me but that is how it is in the training set. 

I would choose a threshold of 3, it's the lowest threshold where false_rate drops to 0% and char_acc is at its highest 78.4%, even though success_rate is 0% at every threshold except 0 anyway cause as ive already mentioned the training set we are working with is less than all incompasing. 

* baseline

In [16]:
def evaluate_baseline():
    correct_words = 0
    correct_chars = 0
    total_chars = 0
    misspelled_fixed = 0
    misspelled_total = 0
    correct_broken = 0
    correct_total = 0

    for correct_word, typed_word in test_pairs:
        output = typed_word  

        if output == correct_word:
            correct_words += 1

        for i in range(len(correct_word)):
            total_chars += 1
            if i < len(output) and output[i] == correct_word[i]:
                correct_chars += 1

        if typed_word != correct_word:
            misspelled_total += 1
            if output == correct_word:
                misspelled_fixed += 1
        else:
            correct_total += 1
            if output != correct_word:
                correct_broken += 1

    return {
        "exact": correct_words / len(test_pairs),
        "char": correct_chars / total_chars,
        "success": misspelled_fixed / misspelled_total if misspelled_total else 0,
        "false": correct_broken / correct_total if correct_total else 0,
    }

print(evaluate_baseline())

{'exact': 0.023255813953488372, 'char': 0.7842565597667639, 'success': 0.0, 'false': 0.0}


# Automated test

In [21]:
# 1. empty input
assert fix_text("") == "", "empty input should return empty string"

# 2. one-letter word
result = fix_text("a")
assert isinstance(result, str), "one-letter word should return a string, not crash"

# 3. capitalization/ case normalization
assert fix_text("HELLO") == fix_text("hello"), "capitalization should be normalized before decoding"

# 4. punctuation/ unsupported character 
result = fix_text("hello, world!")
assert isinstance(result, str), "punctuation should not crash the program"

# 5. transition/emission that never occurred in training data 
result = viterbi("qqqqq")  
assert isinstance(result[0], str), "unseen transitions/emissions should still decode via smoothing, not crash"

# 6. threshold prevents a change
result = threshold("liquify", 5)  
assert result == "liquify", "high threshold should block a correction that would otherwise apply"

print("All tests passed!")

All tests passed!
